# 03 — Preprocessing Pipeline (v2)

**Owner:** Data Preparation & Pipeline

**Goal:** turn raw loan records into clean, numeric model input, using one reusable sklearn `Pipeline` that the whole team shares.

The pipeline code lives in `preprocessing.py` (same folder). This notebook explains each decision and checks that the pipeline works. **Only `train.csv` is used here.**

**Version history**
- **v1:** basic cleaning, imputation, scaling and encoding
- **v2 (this version):** updated with the findings of the EDA (notebook 02). See the "What changed after EDA" section.

## What a pipeline is, and why we use one

A pipeline is a **recipe**: an ordered list of preparation steps saved as a single object. Instead of cleaning the data by hand and saving a "clean" file, we write the steps down once, and the pipeline applies them identically every time.

It has two operations:
- **`fit`**: *learn* whatever the steps need, from the training data only (e.g. the median FICO score used to fill gaps, the mean and std used for scaling, the default rate of each state).
- **`transform`**: *apply* the steps, using what was learned.

**Why this matters (data leakage):** if we computed medians or scaling statistics on all the data, information from the test set would leak into training and inflate our score. With a pipeline, `fit` only ever sees training data. Inside cross-validation, it is re-fitted on each training fold automatically.

**Why it helps the team:** everyone imports the same `build_preprocessor()`, so every model and the final test evaluation use identical preparation.

## What changed after EDA (v1 → v2)

| Change | EDA evidence | Why it matters |
|---|---|---|
| **Removed `has_desc`** | Share of loans with a description: 85% (2009) → 7% (2014) → 0% (2015) | The flag would be all 0 in the 2016 test set. The model would learn a pattern that no longer exists. |
| **Added `loan_to_income`** = loan amount / income | Default rate rises from 13% to 27% across its quintiles; strongest numeric correlation with default (ρ = 0.14) | Captures how heavy the loan is *for this borrower* in one interpretable number (see smoke test for its effect on the score). |
| **`addr_state`: one-hot → smoothed target encoding** | Extreme state default rates come from tiny states (e.g. ID with 1 loan) | 50 one-hot columns mostly fit noise. Target encoding replaces each state with its default rate, shrunk toward the overall rate when the state has few loans. |

Confirmed by EDA and already in v1: `emp_length = "NI"` is informative (23.7% vs 18.2% default), so we keep a "was missing" flag instead of silently imputing it.

In [ ]:
import pandas as pd
from sklearn import set_config
from preprocessing import load_split, clean, build_preprocessor

set_config(display="diagram")   # shows the pipeline as a diagram

X_train, y_train = load_split("../data/train.csv")
print(X_train.shape)
X_train.head(3)

(150000, 14)


,id,issue_d,revenue,dti_n,loan_amnt,fico_n,experience_c,emp_length,purpose,home_ownership_n,addr_state,zip_code,title,desc
0,73582,2007-06-01,180000.0,10.00,3500,687.0,0,< 1 year,other,RENT,NY,100xx,Wedding coming up,NaN
1,83489,2007-06-01,6500.0,6.46,2600,722.0,0,3 years,other,MORTGAGE,CO,806xx,Credit card refinancing,NaN
2,77757,2007-06-01,20000.0,10.00,3000,712.0,0,< 1 year,other,OWN,MA,024xx,Home improvement,NaN


## Step 1 — `clean`: fix data-entry quirks and build row-level features

These are fixed **rules**, not statistics learned from data, so they can be applied row by row with no leakage risk.

| Column | Problem | Fix |
|---|---|---|
| `emp_length` | Text ("10+ years"); "NI" = no information | Convert to 0–10 years; NI → missing (flagged later) |
| `dti_n` | 999 is an undocumented placeholder; a few values above 100% | 999 → missing; cap at 100 |
| `loan_to_income` | (new feature) | `loan_amnt / revenue` |
| `id`, `issue_d`, `experience_c`, `title`, `zip_code`, `desc` | Not useful, not available at test time, or not appropriate | Dropped (reasons in `preprocessing.py`) |

Dropped here means "not given to the model". The columns stay in the data files.

**Note on `dti_n`:** our training sample (2007–2015) contains no 999s and no values above 100. These appear mostly in loans from 2016 onwards, and the test set has 1 placeholder and 11 values above 100. The rule is kept so the pipeline handles them correctly at evaluation time. This is a good example of why cleaning rules belong in the pipeline, not in a one-off edit of the training file.

In [ ]:
cleaned = clean(X_train)
print("Columns after clean:", list(cleaned.columns))
print("Missing values:", cleaned.isna().sum()[lambda s: s > 0].to_dict())
print("\nloan_to_income summary:")
print(cleaned["loan_to_income"].describe().round(3).to_string())

Columns after clean: ['revenue', 'dti_n', 'loan_amnt', 'fico_n', 'emp_length', 'purpose', 'home_ownership_n', 'addr_state', 'loan_to_income']
Missing values: {'emp_length': 7787}

loan_to_income summary:
count    150000.000
mean          0.217
std           0.111
min           0.001
25%           0.130
50%           0.203
75%           0.294
max           0.830


## Step 2 — column-specific transformations

| Group | Columns | Steps | Why |
|---|---|---|---|
| Numeric | `fico_n`, `dti_n`, `emp_length` | Fill missing with **median** + add a "was missing" flag → standardize | Median is robust to outliers; the flag lets the model use the fact that NI borrowers default more |
| Skewed | `revenue`, `loan_amnt`, `loan_to_income` | **log(1+x)** → standardize | Long right tails (income skew ≈ 46 → 0.18 after log) |
| One-hot | `purpose`, `home_ownership_n` | One-hot; levels with < 500 loans grouped as "infrequent" | Models need numbers; rare levels (e.g. `OTHER` home ownership, 54 loans) are unstable on their own |
| Target-encoded | `addr_state` | Replace each state by its **smoothed** default rate | One number instead of 50 columns; small states are pulled toward the overall rate |

**Target encoding and leakage:** target encoding uses the label, so done naively it leaks. sklearn's `TargetEncoder` avoids this with internal **cross-fitting**: during `fit_transform`, each row's encoding is computed from the *other* folds, never from its own label.

**Standardizing** (mean 0, std 1) puts features on the same scale. Distance- and gradient-based models (kNN, SVM, logistic regression) need this. Tree models don't, but it does them no harm.

In [ ]:
preprocessor = build_preprocessor()
preprocessor   # click the boxes in the diagram to expand each step

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('clean', ...), ('columns', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function cle...x7f53582007c0>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",None
,"kw_args kw_args: dict, default=NoneDictionary of ad

## Fit on the training set and inspect the output

In [ ]:
X_train_prepared = preprocessor.fit_transform(X_train, y_train)   # y is needed by the target encoder
feature_names = preprocessor.named_steps["columns"].get_feature_names_out()

print("Raw shape:     ", X_train.shape)
print("Prepared shape:", X_train_prepared.shape)
print("Any missing left?", pd.isna(X_train_prepared).any())

prepared_df = pd.DataFrame(X_train_prepared, columns=feature_names)
prepared_df.head()

Raw shape:      (150000, 14)
Prepared shape: (150000, 24)
Any missing left? False


,num__fico_n,num__dti_n,num__emp_length,num__missingindicator_emp_length,log__revenue,log__loan_amnt,log__loan_to_income,cat__purpose_car,cat__purpose_credit_card,cat__purpose_debt_consolidation,...,cat__purpose_moving,cat__purpose_other,cat__purpose_small_business,cat__purpose_vacation,cat__purpose_infrequent_sklearn,cat__home_ownership_n_MORTGAGE,cat__home_ownership_n_OWN,cat__home_ownership_n_RENT,cat__home_ownership_n_infrequent_sklearn,state__addr_state
0,-0.326541,-0.961495,-1.678307,-0.234,1.948756,-1.810652,-1.922982,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.200223
1,0.812634,-1.389502,-0.836807,-0.234,-4.307766,-2.252441,1.594123,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.149670
2,0.487155,-0.961495,-1.678307,-0.234,-2.190563,-2.039763,-0.586896,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.165454
3,-2.767631,-1.157363,-1.678307,-0.234,-3.573203,-3.401307,-0.830587,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.167969
4,-2.279413,-0.650766,-1.397807,-0.234,-0.383279,-2.196352,-1.577352,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.161269


In [ ]:
# What did `fit` learn? These values come from the training set only.
cols = preprocessor.named_steps["columns"]

num = cols.named_transformers_["num"]
print("Medians used to fill missing values:",
      dict(zip(["fico_n", "dti_n", "emp_length"], num.named_steps["simpleimputer"].statistics_.round(2))))

cat = cols.named_transformers_["cat"]
for col, rare in zip(["purpose", "home_ownership_n"], cat.infrequent_categories_):
    print(f"Grouped as infrequent in {col}: {list(rare) if rare is not None else 'none'}")

te = cols.named_transformers_["state"]
states = pd.DataFrame({
    "n_loans": X_train["addr_state"].value_counts(),
    "raw_default_rate": y_train.groupby(X_train["addr_state"]).mean(),
    "encoded_rate": pd.Series(te.encodings_[0], index=te.categories_[0]),
})
print(f"\nSmallest and largest states (overall default rate {y_train.mean():.3f}):")
print(states.sort_values("n_loans").iloc[[0, 1, 2, 3, -2, -1]].round(3).to_string())

Medians used to fill missing values: {'fico_n': np.float64(692.0), 'dti_n': np.float64(17.48), 'emp_length': np.float64(6.0)}
Grouped as infrequent in purpose: ['educational', 'renewable_energy', 'wedding']
Grouped as infrequent in home_ownership_n: ['OTHER']

Smallest and largest states (overall default rate 0.185):
    n_loans  raw_default_rate  encoded_rate
ID        1             0.000         0.183
ME       83             0.133         0.161
ND       83             0.108         0.150
NE      171             0.275         0.242
NY    12532             0.200         0.200
CA    22060             0.177         0.177


**Smoothing at work:** each state's encoding is `(n × state rate + 100 × overall rate) / (n + 100)`. A state with 1 loan (ID) gets almost exactly the overall rate instead of 0% or 100%. Small states are pulled strongly toward the average, while large states like CA keep (almost) their own rate. With little evidence, the encoder trusts the overall average.

## Smoke test: does the pipeline plug into a model correctly?

This is **not** the baseline model (that belongs to the modeling notebooks). It only checks that preprocessing + model run end to end inside cross-validation, with the preprocessor re-fitted on every fold.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

smoke_model = make_pipeline(build_preprocessor(), LogisticRegression(max_iter=1000))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(smoke_model, X_train, y_train, cv=cv, scoring="roc_auc")
print(f"v2 ROC-AUC: {scores.mean():.3f} ± {scores.std():.3f}   (v1 was 0.660 ± 0.004)")

v2 ROC-AUC: 0.661 ± 0.004   (v1 was 0.660 ± 0.004)


**Why the score barely moved, and why v2 is still the right version**

- **Removing `has_desc` is about the test set, not the CV score.** Cross-validation here mixes loans from 2007–2015, where descriptions still exist, so CV cannot see the problem. On 2016 loans the flag would be all 0. v2 trades nothing in CV for a model that is valid on the future data it will actually score.
- **`loan_to_income` does not add predictive power on its own.** We checked: a gradient-boosting model scores 0.662 ROC-AUC with or without it. The ratio is built from `revenue` and `loan_amnt`, which are already in the model, so the model can find the same information by combining them (for logistic regression, log(loan / income) is literally log(loan) − log(income)). We keep it because it is the most **interpretable** risk signal for the business story (default rate 13% → 27% across its quintiles), and it costs nothing in performance. Lesson: a feature that looks strong in EDA is not necessarily *new* information for the model.

## How teammates use this

```python
from preprocessing import load_split, build_preprocessor
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier

X_train, y_train = load_split("../data/train.csv")
model = make_pipeline(build_preprocessor(), RandomForestClassifier())
# pass `model` straight into cross_val_score / RandomizedSearchCV
```

**Rules**
1. Always pass the **raw** `X_train` (and `y_train`) to the full pipeline. Never pre-clean the data separately.
2. Never call `fit` on test data. The test set is only used once, at the end, with `model.predict`.
3. Suggest changes to `preprocessing.py` through the pipeline owner, so everyone stays on the same version.

## Not done, and why
- **Capping income at the 99.9th percentile** (EDA suggestion): after the log transform, income skew is already 0.18, so the cap would change very little. Can be added if linear models show instability.
- **FICO bands**: tree models find their own cut points; bands would only help linear models. Left to the modeling owners to test.